In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
from sklearn.preprocessing import StandardScaler
import time
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import RidgeClassifier
#from sklearn.naive_bayes import GaussianNB -- cannot be used in RFE
#from sklearn.neighbors import KNeighborsClassifier  -- cannot be used in RFE


In [8]:
##Independent variable - x ; Dependent variable- y; Number of featues - n/k
  #NB &  KNN -- cannot be used in FeatureImportance

def FI(x,y,n):

    FIlist=[]   
    #scaler = StandardScaler()
    #x = pd.DataFrame(scaler.fit_transform(x), columns=x.columns)

    RF=RandomForestClassifier(n_estimators=10,criterion='entropy',random_state=0)
    RF.fit(x, y) 
    
    LR=LogisticRegression(solver='saga',max_iter=1000, random_state=42)
    LR.fit(x, y)  
    
    DT=DecisionTreeClassifier(criterion='gini',max_features='sqrt',splitter='best',random_state=0)
    DT.fit(x, y)
    
    SV=SVC(kernel='linear',random_state=0)
    SV.fit(x, y)

    gbm = XGBClassifier(n_estimators=50, random_state=42) ## or GradientBoostingClassifier
    gbm.fit(x, y)
   
    ridge = RidgeClassifier()
    ridge.fit(x, y)

    
    # Combine results into a single comparison DataFrame
    importance_comparison = pd.DataFrame({
        "Feature": x.columns,
        "Random_Forest_MDI": RF.feature_importances_,
        "Decision Tree": DT.feature_importances_,
        "SV": abs(SV.coef_[0]),
        "Gradient_Boosting_MDI": gbm.feature_importances_,
        "LR_Coefficients": abs(LR.coef_[0]),  # Using absolute values for magnitude
        "Ridge_Coefficients": abs(ridge.coef_[0]),  # Using absolute values for magnitude
    })

    print(importance_comparison.to_string(index=False))
    
    for i in ["Random_Forest_MDI","Decision Tree","SV","Gradient_Boosting_MDI", "LR_Coefficients","Ridge_Coefficients"]:
        
        top_n_features = importance_comparison.sort_values(by=i, ascending=False)['Feature'].head(n).tolist()
        FIlist.append(top_n_features)
        print(FIlist)
    return FIlist
    



def split_scalar(x,y):
    x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.25,random_state=0)
    sc=StandardScaler()
    x_train=sc.fit_transform(x_train)
    x_test=sc.transform(x_test)
    return x_train,x_test,y_train,y_test

def cm_prediction(classifier,x_test):
    y_pred=classifier.predict(x_test)
    
    from sklearn.metrics import confusion_matrix
    cm=confusion_matrix(y_test,y_pred)
    
    from sklearn.metrics import classification_report
    report=classification_report(y_test,y_pred)
    
    from sklearn.metrics import accuracy_score
    Accuracy=accuracy_score(y_test,y_pred)
    
    return classifier,Accuracy,report,x_test,y_test,cm

def logistic(x_train,y_train,x_test):
    from sklearn.linear_model import LogisticRegression
    classifier=LogisticRegression(random_state=0)
    classifier.fit(x_train,y_train)
    classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
    return classifier,Accuracy,report,x_test,y_test,cm

def svm_linear(x_train,y_train,x_test):
    from sklearn.svm import SVC
    classifier=SVC(kernel='linear',random_state=0)
    classifier.fit(x_train,y_train)
    classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
    return classifier,Accuracy,report,x_test,y_test,cm

def svm_NL(x_train,y_train,x_test):
    from sklearn.svm import SVC
    classifier=SVC(kernel='rbf',random_state=0)
    classifier.fit(x_train,y_train)
    classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
    return classifier,Accuracy,report,x_test,y_test,cm

def Navie(x_train,y_train,x_test):
    from sklearn.naive_bayes import GaussianNB
    classifier=GaussianNB()
    classifier.fit(x_train,y_train)
    classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
    return classifier,Accuracy,report,x_test,y_test,cm

def knn(x_train,y_train,x_test):
    from sklearn.neighbors import KNeighborsClassifier
    classifier=KNeighborsClassifier(n_neighbors=5,metric='minkowski',p=2)
    classifier.fit(x_train,y_train)
    classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
    return classifier,Accuracy,report,x_test,y_test,cm

def decision(x_train,y_train,x_test):
    from sklearn.tree import DecisionTreeClassifier
    classifier=DecisionTreeClassifier(criterion='entropy',random_state=0)
    classifier.fit(x_train,y_train)
    classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
    return classifier,Accuracy,report,x_test,y_test,cm

def random(x_train,y_train,x_test):
    from sklearn.ensemble import RandomForestClassifier
    classifier=RandomForestClassifier(n_estimators=10 , criterion='entropy', random_state=0)
    classifier.fit(x_train,y_train)
    classifier,Accuracy,report,x_test,y_test,cm=cm_prediction(classifier,x_test)
    return classifier,Accuracy,report,x_test,y_test,cm

def FI_Classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf):
    dataframe=pd.DataFrame(index=['RF','DT','SVM','XGB','LogR','Ridge'],columns=['Logistic','SVMl','SVMnl','KNN','Naive','Decision','Random'])
    for number, index in enumerate(dataframe.index):
        #print("number, index",number, index)
        dataframe['Logistic'][index]=acclog[number]
        dataframe['SVMl'][index]=accsvml[number]
        dataframe['SVMnl'][index]=accsvmnl[number]
        dataframe['KNN'][index]=accknn[number]
        dataframe['Naive'][index]=accnav[number]
        dataframe['Decision'][index]=accdes[number]
        dataframe['Random'][index]=accrf[number]
    return dataframe


        
            

In [9]:
dataset1=pd.read_csv('watson_healthcare_modified.csv',index_col=None)
df2=dataset1
df2

,EmployeeID,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,...,RelationshipSatisfaction,StandardHours,Shift,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,1313919,41,No,Travel_Rarely,1102,Cardiology,1,2,Life Sciences,1,...,1,80,0,8,0,1,6,4,0,5
1,1200302,49,No,Travel_Frequently,279,Maternity,8,1,Life Sciences,1,...,4,80,1,10,3,3,10,7,1,7
2,1060315,37,Yes,Travel_Rarely,1373,Maternity,2,2,Other,1,...,2,80,0,7,3,3,0,0,0,0
3,1272912,33,No,Travel_Frequently,1392,Maternity,3,4,Life Sciences,1,...,3,80,0,8,3,3,8,7,3,0
4,1414939,27,No,Travel_Rarely,591,Maternity,2,1,Medical,1,...,4,80,1,6,3,3,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1671,1117656,26,Yes,Travel_Rarely,471,Neurology,24,3,Technical Degree,1,...,2,80,0,1,3,1,1,0,0,0
1672,1152327,46,No,Travel_Rarely,1125,Cardiology,10,3,Marketing,1,...,3,80,1,15,3,3,3,2,1,2
1673,1812428,20,No,Travel_Rarely,959,Maternity,1,3,Life Sciences,1,...,4,80,0,1,0,4,1,0,0,0
1674,1812429,39,No,Travel_Rarely,466,Neurology,1,1,Life Sciences,1,...,3,80,1,21,3,3,21,6,11,8


In [13]:
df2=pd.get_dummies(df2,drop_first=True)
##df2['Attrition']
df2.columns

Index(['EmployeeID', 'Age', 'DailyRate', 'DistanceFromHome', 'Education',
       'EmployeeCount', 'EnvironmentSatisfaction', 'HourlyRate',
       'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome',
       'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike',
       'PerformanceRating', 'RelationshipSatisfaction', 'StandardHours',
       'Shift', 'TotalWorkingYears', 'TrainingTimesLastYear',
       'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole',
       'YearsSinceLastPromotion', 'YearsWithCurrManager', 'Attrition_Yes',
       'BusinessTravel_Travel_Frequently', 'BusinessTravel_Travel_Rarely',
       'Department_Maternity', 'Department_Neurology',
       'EducationField_Life Sciences', 'EducationField_Marketing',
       'EducationField_Medical', 'EducationField_Other',
       'EducationField_Technical Degree', 'Gender_Male',
       'JobRole_Administrative', 'JobRole_Nurse', 'JobRole_Other',
       'JobRole_Therapist', 'MaritalStatus_Married', 'MaritalStatus_S

In [14]:
x=df2.drop('Attrition_Yes',axis=1)
y=df2['Attrition_Yes']

In [25]:
FIlist=FI(x,y,35)
FIlist

/Users/Max/opt/anaconda3/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:329: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn("The max_iter was reached which means "


                         Feature  Random_Forest_MDI  Decision Tree           SV  Gradient_Boosting_MDI  LR_Coefficients  Ridge_Coefficients
                      EmployeeID           0.045932       0.045341 1.342178e+00               0.010952     9.096401e-07        3.369904e-08
                             Age           0.078719       0.033150 7.754098e+02               0.045164     6.696990e-07        1.196202e-02
                       DailyRate           0.033167       0.056619 4.400900e+02               0.014938     9.635666e-06        3.899130e-05
                DistanceFromHome           0.042601       0.063690 5.895505e+02               0.024823     2.077980e-07        7.407592e-03
                       Education           0.012645       0.013599 1.541482e+01               0.010235     2.027739e-08        4.402326e-03
                   EmployeeCount           0.000000       0.000000 1.890044e-12               0.000000     4.095256e-09        0.000000e+00
         Environment

[['OverTime_Yes',
  'Age',
  'MonthlyIncome',
  'YearsAtCompany',
  'EmployeeID',
  'DistanceFromHome',
  'NumCompaniesWorked',
  'TotalWorkingYears',
  'MonthlyRate',
  'JobLevel',
  'DailyRate',
  'HourlyRate',
  'EnvironmentSatisfaction',
  'YearsWithCurrManager',
  'JobInvolvement',
  'MaritalStatus_Single',
  'PercentSalaryHike',
  'Shift',
  'WorkLifeBalance',
  'YearsSinceLastPromotion',
  'YearsInCurrentRole',
  'JobSatisfaction',
  'TrainingTimesLastYear',
  'RelationshipSatisfaction',
  'Education',
  'BusinessTravel_Travel_Rarely',
  'JobRole_Other',
  'Department_Maternity',
  'Gender_Male',
  'BusinessTravel_Travel_Frequently',
  'JobRole_Nurse',
  'MaritalStatus_Married',
  'EducationField_Life Sciences',
  'EducationField_Medical',
  'JobRole_Therapist'],
 ['YearsAtCompany',
  'MonthlyIncome',
  'MonthlyRate',
  'DistanceFromHome',
  'DailyRate',
  'OverTime_Yes',
  'EmployeeID',
  'HourlyRate',
  'MaritalStatus_Single',
  'TrainingTimesLastYear',
  'YearsSinceLastPromot

In [26]:
acclog=[]
accsvml=[]
accsvmnl=[]
accknn=[]
accnav=[]
accdes=[]
accrf=[]

In [27]:
for i in FIlist:
    
    df=x[i]
    
    x_train,x_test,y_train,y_test=split_scalar(df,y)

    classifier,Accuracy,report,x_test,y_test,cm=logistic(x_train,y_train,x_test)
    acclog.append(Accuracy)

    classifier,Accuracy,report,x_test,y_test,cm=svm_linear(x_train,y_train,x_test)
    accsvml.append(Accuracy)

    classifier,Accuracy,report,x_test,y_test,cm=svm_NL(x_train,y_train,x_test)
    accsvmnl.append(Accuracy)

    classifier,Accuracy,report,x_test,y_test,cm=Navie(x_train,y_train,x_test)
    accnav.append(Accuracy)

    classifier,Accuracy,report,x_test,y_test,cm=knn(x_train,y_train,x_test)
    accknn.append(Accuracy)

    classifier,Accuracy,report,x_test,y_test,cm=decision(x_train,y_train,x_test)
    accdes.append(Accuracy)

    classifier,Accuracy,report,x_test,y_test,cm=random(x_train,y_train,x_test)
    accrf.append(Accuracy)
    


In [28]:
result=FI_Classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf)


In [ ]:
##result # n=5

In [ ]:
##result # n=6

In [19]:
##result # n=7

,Logistic,SVMl,SVMnl,KNN,Naive,Decision,Random
RF,0.899761,0.890215,0.887828,0.880668,0.871122,0.856802,0.890215
DT,0.892601,0.863962,0.880668,0.875895,0.854415,0.854415,0.887828
SVM,0.871122,0.863962,0.863962,0.861575,0.804296,0.828162,0.875895
XGB,0.890215,0.863962,0.892601,0.880668,0.840095,0.885442,0.880668
LogR,0.863962,0.863962,0.863962,0.875895,0.842482,0.806683,0.871122
Ridge,0.866348,0.863962,0.866348,0.866348,0.797136,0.866348,0.861575


In [24]:
#result # n=30

,Logistic,SVMl,SVMnl,KNN,Naive,Decision,Random
RF,0.940334,0.940334,0.909308,0.880668,0.749403,0.875895,0.897375
DT,0.935561,0.935561,0.916468,0.878282,0.787589,0.890215,0.904535
SVM,0.940334,0.933174,0.911695,0.878282,0.809069,0.871122,0.885442
XGB,0.945107,0.940334,0.923628,0.892601,0.675418,0.880668,0.902148
LogR,0.942721,0.942721,0.911695,0.878282,0.75895,0.868735,0.897375
Ridge,0.930788,0.940334,0.911695,0.883055,0.405728,0.887828,0.885442


In [29]:
##result # n=35

,Logistic,SVMl,SVMnl,KNN,Naive,Decision,Random
RF,0.935561,0.930788,0.911695,0.880668,0.689737,0.885442,0.899761
DT,0.937947,0.933174,0.914081,0.883055,0.424821,0.887828,0.897375
SVM,0.935561,0.928401,0.914081,0.875895,0.699284,0.880668,0.894988
XGB,0.937947,0.930788,0.914081,0.885442,0.680191,0.883055,0.899761
LogR,0.937947,0.933174,0.914081,0.875895,0.696897,0.863962,0.892601
Ridge,0.930788,0.930788,0.916468,0.894988,0.443914,0.887828,0.897375


In [ ]:
## On comparing with various n values for FeatureImportance, n=7 gives better result